# TS-GNN Data Acquisition for Google Colab

This notebook assumes it is located inside your Google Drive project folder (e.g., `MyDrive/Sheaf Neural Networks/ts-gnn/`). It will download all the necessary datasets (Breast Cancer, Colorectal Cancer, Regulatory Priors) exactly to the relative `./data/` folder.

**Instructions:**
1. Mount your Google Drive.
2. Make sure your working directory is correctly set to the `ts-gnn` root folder.
3. Run all cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# ⚠️ IMPORTANT: Change this path to match exactly where your 'ts-gnn' folder is in your Drive
ROOT_DIR = '/content/drive/MyDrive/Sheaf Neural Networks/ts-gnn'
os.chdir(ROOT_DIR)
print(f"Current working directory: {os.getcwd()}")

### Install Dependencies

In [ ]:
!pip install --quiet scanpy anndata GEOparse fair-esm torch torchvision torchaudio

### 1. Download Core Datasets and Priors
First, we run the automated download script which prepares Breast Cancer matrices, Colorectal Cancer matrices, and external priors (RegNetwork, TargetGeneReg, JASPAR, etc.).

In [ ]:
!python src/tsgnn/data/download.py

In [ ]:
import os
from pathlib import Path

string_dir = Path(os.getcwd()) / "data" / "external" / "string"
string_dir.mkdir(parents=True, exist_ok=True)

string_links = string_dir / "9606.protein.links.v12.0.txt.gz"
string_aliases = string_dir / "9606.protein.aliases.v12.0.txt.gz"

# URL primario e mirror di fallback
STRING_LINKS_URLS = [
    "https://stringdb-downloads.org/download/protein.links.v12.0/9606.protein.links.v12.0.txt.gz",
    "https://version-12-0.string-db.org/download/protein.links.v12.0/9606.protein.links.v12.0.txt.gz",
]
STRING_ALIASES_URLS = [
    "https://stringdb-downloads.org/download/protein.aliases.v12.0/9606.protein.aliases.v12.0.txt.gz",
    "https://version-12-0.string-db.org/download/protein.aliases.v12.0/9606.protein.aliases.v12.0.txt.gz",
]

def wget_download(urls, dest: Path):
    if dest.exists() and dest.stat().st_size > 1024:
        print(f"✓ Già presente: {dest.name} ({dest.stat().st_size / 1e6:.0f} MB)")
        return True
    for url in urls:
        print(f"Scaricando da: {url}")
        ret = os.system(
            f'wget -q --show-progress --retry-connrefused --tries=3 '
            f'--timeout=120 -O "{dest}" "{url}"'
        )
        if ret == 0 and dest.exists() and dest.stat().st_size > 1024:
            print(f"✓ {dest.name}: {dest.stat().st_size / 1e6:.0f} MB")
            return True
        print(f"  Mirror fallito, provo il successivo...")
        if dest.exists():
            dest.unlink()
    print(f"✗ Download fallito per {dest.name}. Scarica manualmente da https://string-db.org/")
    return False

print("=== STRING v12 download ===")
wget_download(STRING_LINKS_URLS, string_links)
wget_download(STRING_ALIASES_URLS, string_aliases)

### 2. Download STRING v12 Protein Links (~700 MB)
`download.py` tenta già questo download, ma per file grandi su Colab è più affidabile usare `wget` direttamente. La cella è idempotente: salta se il file esiste già.

### 2. Download Gambardella 2022 (Breast Cancer Cell Line Atlas)
This dataset is crucial for maintaining a pristine ground truth for the model's training phase.

In [ ]:
import os
import requests
from pathlib import Path

def download_file(url, dest):
    print(f"Downloading {url} to {dest}...")
    resp = requests.get(url, stream=True, timeout=300)
    resp.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Done: {dest.stat().st_size / 1e6:.1f} MB")

gamb_dir = Path(os.getcwd()) / "data" / "raw" / "breast_gambardella"
os.makedirs(gamb_dir, exist_ok=True)

# File 1: RAW.UMI.counts.BC.cell.lines.rds
url1 = "https://ndownloader.figshare.com/files/28893384"
dest1 = gamb_dir / "RAW.UMI.counts.BC.cell.lines.rds"
if not dest1.exists(): download_file(url1, dest1)

# File 2: GFICF.processed.counts.gficf
url2 = "https://ndownloader.figshare.com/files/33943715"
dest2 = gamb_dir / "GFICF.processed.counts.gficf.tar"
if not dest2.exists(): download_file(url2, dest2)

### 3. Generate ESM-2 Embeddings for TP53 Alleles
Finally, we generate the mutation embeddings directly into the Google Drive caching folder.

In [ ]:
!python src/tsgnn/data/allele_embeddings.py